In [1]:
!pip install plotly

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 30.5 MB/s eta 0:00:0000:0100:01


In [2]:
!pip install "anywidget>=0.9.13"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.7/213.7 kB 2.7 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 477.3/477.3 kB 7.5 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 2.2 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.3/217.3 kB 4.7 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 16.7 MB/s eta 0:00:0000:0100:01


In [3]:
!pip install -U kaleido

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 6.5 MB/s eta 0:00:00:00:0100:01


In [1]:
from samap.mapping import SAMAP
from samap.analysis import (get_mapping_scores, GenePairFinder, transfer_annotations,
                            sankey_plot, chord_plot, CellTypeTriangles, 
                            ParalogSubstitutions, FunctionalEnrichment,
                            convert_eggnog_to_homologs, GeneTriangles)
from samalg import SAM
import pandas as pd
from Bio import SeqIO
from samap.utils import (save_samap, load_samap)
import scanpy as sc
import matplotlib.colors
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from scipy import sparse 
from scipy import cluster
import seaborn as sns
import random
import sklearn
from scipy.stats import poisson
from sklearn.neighbors import KernelDensity
import time
import dill
from scipy.optimize import minimize
import pickle
import itertools
import os
import plotly.express as px

In [2]:
df_NN = pd.DataFrame(columns = ['org','celltype','mapping','frac cells'])

In [3]:
a = 0

In [56]:
#repeat blocks 4 - 20 for each species
sm = load_samap('../../Active_SAMap_Joined/sm_Allen_nonneural2000_dr_NONneuron_090720206.pkl')

In [57]:
sam = SAM()
sam.load_data('../../Active_SAM_joined/SAM_DR_ncbi_joined_cleaned_07172026.h5ad')

In [58]:
ref = 'mg'

In [59]:
org = 'dr'

In [60]:
ref_level = 'figure2e_mapping'
org_level = 'ss_subclass_nounlabeled_nmm_cl_v5_nn'

In [61]:
sm.sams[org].adata.obs['ss_subclass_nounlabeled_nmm_cl_v5_nn'] = sam.adata.obs['ss_subclass_nounlabeled_nmm_cl_v5_nn']

In [62]:
#This maps the mouse cell types to the combined cell types
mappings = {'316 Bergmann NN':'339 Astrocyte-like NN',
'317 Astro-CB NN':'339 Astrocyte-like NN',
'318 Astro-NT NN':'339 Astrocyte-like NN',
'319 Astro-TE NN':'339 Astrocyte-like NN',
'320 Astro-OLF NN':'339 Astrocyte-like NN',
'334 Microglia NN':'340 Macrophage NN',
'335 BAM NN':'340 Macrophage NN'}

In [63]:
fin_ref = []
for item in sm.sams[ref].adata.obs['subclass_id_label']:
    if item in mappings:
        fin_ref.append(mappings[item])
    else:
        fin_ref.append(item)

In [64]:
sm.sams[ref].adata.obs['figure2e_mapping'] = fin_ref

In [65]:
keys = {ref:ref_level,org:org_level}
D,MappingTable = get_mapping_scores(sm,keys)

lim_MappingTable = MappingTable.filter(like=org)
lim_MappingTable = lim_MappingTable[lim_MappingTable.index.str.contains(ref)]

In [66]:
for item in sm.sams[ref].adata.obs[ref_level].unique():
    print(org + '_' + item)
    if org + '_' + item in lim_MappingTable.columns:
        df_NN.loc[a, 'mapping'] = lim_MappingTable.loc[ref + '_' + item, org + '_' + item]
        df_NN.loc[a, 'org'] = org
        df_NN.loc[a, 'celltype'] = item
        df_NN.loc[a, 'num cells'] = len(sm.sams[org].adata[sm.sams[org].adata.obs[org_level] == item])
        df_NN.loc[a, 'frac cells'] = len(sm.sams[org].adata[sm.sams[org].adata.obs[org_level] == item])/len(sam.adata)
        a += 1

dr_339 Astrocyte-like NN
dr_322 Tanycyte NN
dr_321 Astroependymal NN
dr_325 CHOR NN
dr_330 VLMC NN
dr_323 Ependymal NN
dr_324 Hypendymal NN
dr_326 OPC NN
dr_327 Oligo NN
dr_329 ABC NN
dr_328 OEC NN
dr_331 Peri NN
dr_332 SMC NN
dr_333 Endo NN
dr_338 Lymphoid NN
dr_340 Macrophage NN
dr_337 DC NN
dr_336 Monocytes NN


In [67]:
df_NN

,org,celltype,mapping,frac cells,num cells
0,cj,339 Astrocyte-like NN,0.83685,0.071484,5344.0
1,cj,322 Tanycyte NN,0.731423,0.004882,365.0
2,cj,325 CHOR NN,0.79653,0.000375,28.0
3,cj,330 VLMC NN,0.921882,0.005765,431.0
4,cj,323 Ependymal NN,0.875249,0.00325,243.0
5,cj,326 OPC NN,0.938713,0.031328,2342.0
6,cj,327 Oligo NN,0.80241,0.123906,9263.0
7,cj,329 ABC NN,0.913082,0.001859,139.0
8,cj,332 SMC NN,0.752329,0.000482,36.0
9,cj,333 Endo NN,0.972333,0.000522,39.0


In [68]:
df_NN[df_NN['celltype'] == '330 VLMC NN']

,org,celltype,mapping,frac cells,num cells
3,cj,330 VLMC NN,0.921882,0.005765,431.0
14,ac,330 VLMC NN,0.944588,0.011712,561.0
24,xt,330 VLMC NN,0.929616,0.001871,79.0
33,dr,330 VLMC NN,0.845929,0.005811,357.0


In [69]:
df_NN[df_NN['celltype'] == '333 Endo NN']

,org,celltype,mapping,frac cells,num cells
9,cj,333 Endo NN,0.972333,0.000522,39.0
19,ac,333 Endo NN,0.936344,0.00142,68.0
28,xt,333 Endo NN,0.958431,0.001492,63.0
41,dr,333 Endo NN,0.852728,0.021781,1338.0


In [70]:
df_NN[df_NN['celltype'] == '329 ABC NN']

,org,celltype,mapping,frac cells,num cells
7,cj,329 ABC NN,0.913082,0.001859,139.0
18,ac,329 ABC NN,0.92585,0.000585,28.0
27,xt,329 ABC NN,0.753994,0.000782,33.0
37,dr,329 ABC NN,0.617786,0.002588,159.0


In [71]:
df_NN[df_NN['celltype'] == '340 Macrophage NN']

,org,celltype,mapping,frac cells,num cells
10,cj,340 Macrophage NN,0.854496,0.00309,231.0
21,ac,340 Macrophage NN,0.917937,0.018246,874.0
30,xt,340 Macrophage NN,0.882048,0.001729,73.0
43,dr,340 Macrophage NN,0.551516,0.031727,1949.0


In [72]:
#repeat till here for each species first before you move on
set.intersection(*(set(g['celltype']) for _, g in df_NN.groupby('org')))

{'326 OPC NN',
 '327 Oligo NN',
 '329 ABC NN',
 '330 VLMC NN',
 '333 Endo NN',
 '339 Astrocyte-like NN',
 '340 Macrophage NN'}

In [73]:
#Tanycyte added as a cell type lost in zebrafish
ct_int = ['322 Tanycyte NN','326 OPC NN','327 Oligo NN','339 Astrocyte-like NN','340 Macrophage NN']

In [74]:
df_NN = df_NN[df_NN['celltype'].isin(ct_int)]

In [75]:
df_NN

,org,celltype,mapping,frac cells,num cells
0,cj,339 Astrocyte-like NN,0.83685,0.071484,5344.0
1,cj,322 Tanycyte NN,0.731423,0.004882,365.0
5,cj,326 OPC NN,0.938713,0.031328,2342.0
6,cj,327 Oligo NN,0.80241,0.123906,9263.0
10,cj,340 Macrophage NN,0.854496,0.00309,231.0
11,ac,339 Astrocyte-like NN,0.855443,0.032234,1544.0
12,ac,322 Tanycyte NN,0.520261,0.002902,139.0
16,ac,326 OPC NN,0.837983,0.02785,1334.0
17,ac,327 Oligo NN,0.722263,0.12405,5942.0
21,ac,340 Macrophage NN,0.917937,0.018246,874.0


In [76]:
#this allows us to get the dot size for the legend
test = [0,0.05,0.1]
org = []
fin_ct = []
frac = []
mapping = []
numcells = []
for th in test:
    for item in df_NN['celltype'].unique():
        org.append('test_' + str(th))
        fin_ct.append(item)
        frac.append(th)
        mapping.append(1)
        numcells.append(0)

In [77]:
test_df = pd.DataFrame([org,fin_ct,mapping,frac,numcells], index = ['org','celltype','mapping','frac cells','num cells']).T

In [78]:
test_df

,org,celltype,mapping,frac cells,num cells
0,test_0,339 Astrocyte-like NN,1,0,0
1,test_0,322 Tanycyte NN,1,0,0
2,test_0,326 OPC NN,1,0,0
3,test_0,327 Oligo NN,1,0,0
4,test_0,340 Macrophage NN,1,0,0
5,test_0.05,339 Astrocyte-like NN,1,0.05,0
6,test_0.05,322 Tanycyte NN,1,0.05,0
7,test_0.05,326 OPC NN,1,0.05,0
8,test_0.05,327 Oligo NN,1,0.05,0
9,test_0.05,340 Macrophage NN,1,0.05,0


In [79]:
df_NN = pd.concat([df_NN,test_df])

In [80]:
df_NN

,org,celltype,mapping,frac cells,num cells
0,cj,339 Astrocyte-like NN,0.83685,0.071484,5344.0
1,cj,322 Tanycyte NN,0.731423,0.004882,365.0
5,cj,326 OPC NN,0.938713,0.031328,2342.0
6,cj,327 Oligo NN,0.80241,0.123906,9263.0
10,cj,340 Macrophage NN,0.854496,0.00309,231.0
11,ac,339 Astrocyte-like NN,0.855443,0.032234,1544.0
12,ac,322 Tanycyte NN,0.520261,0.002902,139.0
16,ac,326 OPC NN,0.837983,0.02785,1334.0
17,ac,327 Oligo NN,0.722263,0.12405,5942.0
21,ac,340 Macrophage NN,0.917937,0.018246,874.0


In [81]:
df_NN['frac cells'] = df_NN['frac cells'].clip(upper=.1)

In [82]:
df_NN['frac cells'] = df_NN['frac cells'].astype('float')
df_NN['mapping'] = df_NN['mapping'].astype('float')

In [83]:
fig = px.scatter(df_NN, x = 'org', y = 'celltype', size = 'frac cells', color = 'mapping', color_continuous_scale= 'Blues', range_color=[0,.99],opacity = 1)
fig.update_layout(font=dict(family="Helvetica, Arial, sans-serif", color="black"))
fig.update_xaxes(categoryorder='array', categoryarray= ['cj','ac','xt','dr'])
fig.update_yaxes(categoryorder='array', categoryarray= ['322 Tanycyte NN','340 Macrophage NN','339 Astrocyte-like NN','326 OPC NN','327 Oligo NN',])
fig.update_layout(
    autosize=False,
    width=700,
    height=300,
)

fig.write_image("../../Figures/Figures_09202026/NN_mapping_allorgs_09132026.pdf")
fig.write_image("../../Figures/Figures_09202026/NN_mapping_allorgs_09132026.png")
fig.write_image("../../Figures/Figures_09202026/NN_mapping_allorgs_09132026.svg")